<a href="https://colab.research.google.com/github/sashachereshnya-pixel/compling2025/blob/main/%D0%A7%D0%95%D0%A0%D0%95%D0%92%D0%98%D0%9A_%D0%94%D0%97_fine_tuning_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [5]:
import numpy as np
from datasets import load_dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
import evaluate

# Загрузка датасета
dataset = load_dataset("ag_news")

# Загрузка токенизатора и модели
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4)

# Токенизация данных
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Настройка аргументов для обучения
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

# Метрика accuracy
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    return {"accuracy": acc["accuracy"]}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

# Оценка на тестовой выборке
test_results = trainer.evaluate()
print("Test Accuracy:", test_results["eval_accuracy"])

# Сохранение модели
model.save_pretrained("./ag_news_model")
tokenizer.save_pretrained("./ag_news_model")

# Тестирование модели на новых новостях
from transformers import pipeline

classifier = pipeline("text-classification", model="./ag_news_model", tokenizer="./ag_news_model")

new_texts = [
    "The stock market is experiencing significant fluctuations.",
    "The local team won the championship last night.",
    "Scientists have discovered a new species of frog in the Amazon."
]

for text in new_texts:
    result = classifier(text)
    print(f"Text: {text}\nPredicted class: {result[0]['label']}, Confidence: {result[0]['score']}\n")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.200713,0.180081,0.942368
2,0.133675,0.185998,0.947763
3,0.088220,0.218506,0.945658


...

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Test Accuracy: 0.9478947368421052


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: The stock market is experiencing significant fluctuations.
Predicted class: LABEL_2, Confidence: 0.9818618297576904

Text: The local team won the championship last night.
Predicted class: LABEL_1, Confidence: 0.9945534467697144

Text: Scientists have discovered a new species of frog in the Amazon.
Predicted class: LABEL_3, Confidence: 0.9858270883560181

